# CircuitGraph Quickstart

**Audience:** Python users who want to build circuits programmatically and inspect results as Pandas and NetworkX objects.

This notebook builds a voltage divider with `CircuitGraph`, compiles it to a Xyce netlist, and optionally runs an operating-point analysis when Xyce is installed.

In [ ]:
from pathlib import Path
import shutil
from tempfile import TemporaryDirectory

from xyce_py import CircuitGraph, NetlistCompiler, Resistor, VoltageSource, find_xyce_executable


def xyce_available() -> bool:
    default_path = Path("/usr/local/XyceNF_7.10/bin/Xyce")
    return default_path.exists() or shutil.which("Xyce") is not None

## Build the topology

`CircuitGraph` is the public topology input. It owns an internal `networkx.MultiDiGraph` because circuits need parallel branches and directed terminal polarity.

In [ ]:
graph = CircuitGraph(xyce_path="Xyce")
graph.add_node("gnd", is_ground=True)
graph.add_branch("vin", "gnd", [VoltageSource("supply", 10.0)])
graph.add_branch("vin", "vout", [Resistor("top", 1000)])
graph.add_branch("vout", "gnd", [Resistor("bottom", 1000)])

assert graph.G.nodes["gnd"]["is_ground"] is True
assert graph.G.number_of_edges() == 3

## Compile without running Xyce

Compile-only workflows are useful for reviewing the generated netlist before a simulation run.

In [ ]:
netlist = NetlistCompiler(graph.G, graph.spice_directives).compile()
print(netlist)

## Optional operating-point run

The result keeps the waveform table as the canonical numeric output. `solved_graph()` returns a copied topology annotated with selected node voltages.

In [ ]:
if xyce_available():
    with TemporaryDirectory() as tmpdir:
        run_graph = CircuitGraph(xyce_path=find_xyce_executable(), base_out_dir=tmpdir)
        run_graph.add_node("gnd", is_ground=True)
        run_graph.add_branch("vin", "gnd", [VoltageSource("supply", 10.0)])
        run_graph.add_branch("vin", "vout", [Resistor("top", 1000)])
        run_graph.add_branch("vout", "gnd", [Resistor("bottom", 1000)])

        result = run_graph.simulate_op()
        print(result.translated_waveforms())

        solved = result.solved_graph(row=0)
        print("vout solved voltage:", solved.nodes["vout"]["solved_voltage"])
else:
    print("Xyce was not found; the compile-only cells above are still runnable.")